# Practice 2 — Chạy lại 3 cấu hình tốt nhất 20 epoch trên GPU Colab T4

Chạy lại đúng 3 cấu hình tốt nhất trong `results/TRAINING_RESULTS.md` rồi chấm test để ra `confusion_matrix.png` và `per_class_f1.png` mới:
- ResNet18, fine-tune, Adam, lr 0.0001, batch 32 (val 0,8456)
- VGG11, fine-tune, Adam, lr 0.0001, batch 32 (val 0,8538)
- DenseNet121, fine-tune, Adam, lr 0.0005, batch 64 (val 0,8618)

Chung: 20 epoch, ảnh 64, optimizer Adam, strategy fine_tune, có AMP trên T4.

Chạy từng ô lệnh từ trên xuống dưới, đừng chạy tắt.

## 1. Kiểm tra GPU (phải hiện Tesla T4 mới đúng)

In [ ]:
import torch
assert torch.cuda.is_available(), "Chưa bật GPU: chọn menu Runtime > Change runtime type > T4 GPU rồi chạy lại ô này."
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("AMP bật:", True, "(trainer tự bật autocast khi device=cuda)")

## 2. Lấy code từ GitHub

In [ ]:
REPO_URL = "https://github.com/Sper-il/Lab2.git"
import os
if not os.path.isdir("/content/Lab2"):
    !git clone $REPO_URL /content/Lab2
else:
    !git -C /content/Lab2 pull --ff-only
%cd /content/Lab2
!git log --oneline -3
!git status --short | head -20

## 3. Cài thư viện (giữ nguyên torch của Colab để không lỗi CUDA trên T4)

In [ ]:
!pip -q install tensorboard pyyaml matplotlib pillow numpy
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4. Tải CIFAR-10 về máy Colab

In [ ]:
!python scripts/download_data.py
!python scripts/verify_data.py

## 5. Chạy lại ResNet18 tốt nhất: lr 0.0001, batch 32, 20 epoch

In [ ]:
!python -m src.training --config configs/train_config.yaml --models resnet18 --lr 0.0001 --batch-size 32 --epochs 20 --image-size 64 --strategy fine_tune --optimizer adam

## 6. Chạy lại VGG11 tốt nhất: lr 0.0001, batch 32, 20 epoch

In [ ]:
!python -m src.training --config configs/train_config.yaml --models vgg11 --lr 0.0001 --batch-size 32 --epochs 20 --image-size 64 --strategy fine_tune --optimizer adam

## 7. Chạy lại DenseNet121 tốt nhất: lr 0.0005, batch 64, 20 epoch (lâu nhất, khoảng 20–30 phút trên T4)

In [ ]:
!python -m src.training --config configs/train_config.yaml --models densenet121 --lr 0.0005 --batch-size 64 --epochs 20 --image-size 64 --strategy fine_tune --optimizer adam

## 8. Chấm test cả 3 run mới (ra metrics.json + confusion_matrix.png + per_class_f1.png)

In [ ]:
import glob, json, os, subprocess, time
cutoff = time.time() - 24 * 3600  # chỉ chấm run mới trong 24 giờ qua
ckpts = []
for pat in ["results/runs/*/resnet18-*/checkpoints/best.pt", "results/runs/*/vgg11-*/checkpoints/best.pt", "results/runs/*/densenet121-*/checkpoints/best.pt"]:
    ckpts += sorted(glob.glob(pat))
print("Tìm thấy checkpoint:", ckpts)
fresh = [c for c in ckpts if os.path.getmtime(c) >= cutoff]
print("Chấm", len(fresh), "run mới:", fresh)
assert fresh, "Không thấy run mới nào. Kiểm tra lại bước train ở trên."
for ckpt in fresh:
    print("=" * 80)
    print(ckpt)
    subprocess.run(["python", "-m", "src.evaluation", "--checkpoint", ckpt], check=True)
print("Xong chấm test.")

## 9. In điểm test + hiển thị confusion_matrix.png và per_class_f1.png ngay trên Colab

In [ ]:
import glob, json, os, time
from IPython.display import Image, display
cutoff = time.time() - 24 * 3600
for mpath in sorted(glob.glob("results/runs/*/resnet18-*/eval/metrics.json")) + sorted(glob.glob("results/runs/*/vgg11-*/eval/metrics.json")) + sorted(glob.glob("results/runs/*/densenet121-*/eval/metrics.json")):
    if os.path.getmtime(mpath) < cutoff:
        continue
    with open(mpath, encoding="utf-8") as f:
        m = json.load(f)
    print("=" * 80)
    print("Model:", m.get("model"), "| checkpoint:", m.get("checkpoint"))
    print(f"Test accuracy: {m['test']['accuracy']:.4f} | macro-F1: {m['test']['macro_f1']:.4f}")
    base = os.path.dirname(mpath)
    for png in [os.path.join(base, "confusion_matrix.png"), os.path.join(base, "per_class_f1.png")]:
        assert os.path.isfile(png), f"Thiếu file: {png}. Ô chấm test chưa chạy xong."
        print(png)
        display(Image(filename=png))

## 10. Nén kết quả để tải về (dùng tarfile của Python, không lỗi biến shell)

In [ ]:
import glob, os, tarfile, time
new_runs = [d for d in glob.glob("results/runs/*") if (time.time() - os.path.getmtime(d)) < 24 * 3600]
print("run mới:", new_runs)
assert new_runs, "Không có run mới để nén."
out = "/content/best3-results.tar.gz"
with tarfile.open(out, "w:gz") as tar:
    for d in sorted(new_runs):
        tar.add(d, arcname=os.path.relpath(d, "."))
print("Xong:", out)
print("Tải file này ở panel Files bên trái, giải nén vào results/runs/ trong máy bạn.")